#### **Advanced architecture patterns**

To make our model more effective we should apply the following concepts: (1) Batch Normalization, (2) Depthwise separable convolution and (3) Model ensembling. Let's discuss them one-by-one.

#### **Batch Normalization**

It is a type of layer introduced in 2015 by Ioffe and Szegedy (https://arxiv.org/abs/1502.03167) which can adaptively normalize data even as the mean and variance change over time during training. It helps with gradient propagation and thus allows fro deeper networks.

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Preprocess the data
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Define the model with Batch Normalization
def build_model():
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
        layers.BatchNormalization(), # Batch Normalization layer
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
        layers.BatchNormalization(), # Batch Normalization layer
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ])
    return model
#https://arxiv.org/pdf/1706.02515
model = build_model()
model.summary()

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
print("\nTraining the model...")
history = model.fit(x_train, y_train, batch_size=128, epochs=5, validation_split=0.1)

# Evaluate the model
print("\nEvaluating the model...")
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

2026-06-21 16:47:36.202100: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782060456.396104      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782060456.450581      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782060456.874835      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782060456.874916      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782060456.874919      58 computation_placer.cc:177] computation placer alr

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


I0000 00:00:1782060470.857162      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 26, 26, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 11, 11, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...


Training the model...
Epoch 1/5


I0000 00:00:1782060474.150939     129 service.cc:152] XLA service 0x794ac000be20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782060474.150974     129 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1782060474.499340     129 cuda_dnn.cc:529] Loaded cuDNN version 91002


 45/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5321 - loss: 1.8769

I0000 00:00:1782060477.175031     129 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


422/422 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9167 - loss: 0.2912 - val_accuracy: 0.9043 - val_loss: 0.2999
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9707 - loss: 0.0974 - val_accuracy: 0.9873 - val_loss: 0.0455
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9778 - loss: 0.0710 - val_accuracy: 0.9882 - val_loss: 0.0450
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9810 - loss: 0.0616 - val_accuracy: 0.9872 - val_loss: 0.0449
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9841 - loss: 0.0525 - val_accuracy: 0.9883 - val_loss: 0.0455

Evaluating the model...
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9873 - loss: 0.0392
Test Loss: 0.0392
Test Accuracy: 0.9873


#### **Batch Renormalization** (https://arxiv.org/pdf/1702.03275)

Keras 3 doesn't support the `renorm=True` argument passing in the `BatchNormalization` function so we write the class with the help of ChatGPT....

In [4]:
import tensorflow as tf
from keras import layers

class BatchRenormalization(layers.Layer):
    def __init__(
        self,
        momentum=0.99,
        epsilon=1e-3,
        rmax=3.0,
        dmax=5.0,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.momentum = momentum
        self.epsilon = epsilon
        self.rmax = rmax
        self.dmax = dmax

    def build(self, input_shape):

        dim = input_shape[-1]

        self.gamma = self.add_weight(
            shape=(dim,),#https://arxiv.org/pdf/1706.02515
            initializer="ones",
            trainable=True,
            name="gamma"
        )

        self.beta = self.add_weight(
            shape=(dim,),
            initializer="zeros",
            trainable=True,
            name="beta"
        )

        self.moving_mean = self.add_weight(
            shape=(dim,),
            initializer="zeros",
            trainable=False,
            name="moving_mean"
        )

        self.moving_var = self.add_weight(
            shape=(dim,),
            initializer="ones",
            trainable=False,
            name="moving_var"
        )

    def call(self, x, training=False):

        axes = list(range(len(x.shape) - 1))

        if training:

            batch_mean = tf.reduce_mean(x, axis=axes)

            batch_var = tf.reduce_mean(
                tf.square(x - batch_mean),
                axis=axes
            )

            batch_std = tf.sqrt(batch_var + self.epsilon)
            moving_std = tf.sqrt(self.moving_var + self.epsilon)

            r = tf.stop_gradient(batch_std / moving_std)
            d = tf.stop_gradient(
                (batch_mean - self.moving_mean) / moving_std
            )

            r = tf.clip_by_value(r, 1/self.rmax, self.rmax)
            d = tf.clip_by_value(d, -self.dmax, self.dmax)

            x_hat = ((x - batch_mean) / batch_std) * r + d

            self.moving_mean.assign(
                self.momentum * self.moving_mean +
                (1 - self.momentum) * batch_mean
            )

            self.moving_var.assign(
                self.momentum * self.moving_var +
                (1 - self.momentum) * batch_var
            )

        else:

            x_hat = (
                x - self.moving_mean
            ) / tf.sqrt(self.moving_var + self.epsilon)

        return self.gamma * x_hat + self.beta

In [5]:
# Define the model with Batch Renormalization
def build_model_renorm():
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
        BatchRenormalization(), # Batch Renormalization layer
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
        BatchRenormalization(), # Batch Renormalization layer
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ])
    return model

model_renorm = build_model_renorm()
model_renorm.summary()

# Compile the model
model_renorm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
print("\nTraining the model with Batch Renormalization...")
history_renorm = model_renorm.fit(x_train, y_train, batch_size=128, epochs=5, validation_split=0.1)

# Evaluate the model
print("\nEvaluating the model with Batch Renormalization...")
loss_renorm, accuracy_renorm = model_renorm.evaluate(x_test, y_test)
print(f"Test Loss (Batch Renormalization): {loss_renorm:.4f}")
print(f"Test Accuracy (Batch Renormalization): {accuracy_renorm:.4f}")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_renormalization           │ (None, 26, 26, 32)     │           128 │
│ (BatchRenormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_renormalization_1         │ (None, 11, 11, 64)     │           256 │
│ (BatchRenormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,210 (137.54 KB)

 Trainable params: 35,018 (136.79 KB)

 Non-trainable params: 192 (768.00 B)


Training the model with Batch Renormalization...
Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9297 - loss: 0.2358 - val_accuracy: 0.9850 - val_loss: 0.0600
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9711 - loss: 0.1324 - val_accuracy: 0.9888 - val_loss: 0.0545
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9800 - loss: 0.0741 - val_accuracy: 0.9917 - val_loss: 0.0398
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9840 - loss: 0.0545 - val_accuracy: 0.9900 - val_loss: 0.0404
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9859 - loss: 0.0512 - val_accuracy: 0.9922 - val_loss: 0.0346

Evaluating the model with Batch Renormalization...
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9916 - loss: 0.0269
Test Loss (Batch Renormalization): 0.0269
Test Accuracy (Batch Renormalization): 0.9916


Although the paper shows a clear benefit over batch normalization, at no apparent cost. However, this experiment doesn't follow that claim.

#### **Self-Normalizing Neural Networks** (https://arxiv.org/pdf/1706.02515)

In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the SNN model
def build_snn_model():
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Flatten(),
        layers.Dense(256, activation='selu', kernel_initializer='lecun_normal'),
        layers.AlphaDropout(0.1), # AlphaDropout for SNNs
        layers.Dense(128, activation='selu', kernel_initializer='lecun_normal'),
        layers.AlphaDropout(0.1),
        layers.Dense(10, activation='softmax', kernel_initializer='lecun_normal') # Output layer
    ])
    return model

model_snn = build_snn_model()
model_snn.summary()

# Compile the model
model_snn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
print("\nTraining the SNN model...")
history_snn = model_snn.fit(x_train, y_train, batch_size=128, epochs=10, validation_split=0.1)

# Evaluate the model
print("\nEvaluating the SNN model...")
loss_snn, accuracy_snn = model_snn.evaluate(x_test, y_test)
print(f"Test Loss (SNN): {loss_snn:.4f}")
print(f"Test Accuracy (SNN): {accuracy_snn:.4f}")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ alpha_dropout (AlphaDropout)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ alpha_dropout_1 (AlphaDropout)  │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)


Training the SNN model...
Epoch 1/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8750 - loss: 0.4088 - val_accuracy: 0.9508 - val_loss: 0.1747
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9359 - loss: 0.2132 - val_accuracy: 0.9665 - val_loss: 0.1239
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9514 - loss: 0.1579 - val_accuracy: 0.9722 - val_loss: 0.0996
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9602 - loss: 0.1271 - val_accuracy: 0.9697 - val_loss: 0.0997
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9674 - loss: 0.1052 - val_accuracy: 0.9770 - val_loss: 0.0819
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9717 - loss: 0.0894 - val_accuracy: 0.9773 - val_loss: 0.0838
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9750 - loss: 0.0781 - val_accuracy: 0.9778 - val_loss: 0.0833
Epoch 8/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9782 - loss

This is superfast but doesn't match the performance.

#### **Depth-Wise Separable Convolution**

What if there's a layer we can use as a drop-in replacement for `Conv2D` that will make our model lighter (fewer trainable weight parameters) and faster(fewer floating-point operations) and cause it to perform a few percentage points better on its task?

The depth-wise separable convolution layer performs a spatial convolution on each channel of its input, independently, before mixing output channels via a pointwise convolution.

This is equivalent to separating the learning of spatial features and channel-wise features.

In [7]:
### depth-wise separable convolution

from keras.models import Sequential, Model
from keras import layers

height = 28
width = 28
channels = 1

num_classes = 10

model = Sequential()
model.add(layers.SeparableConv2D(28,3,activation='relu',input_shape=(height, width, channels,)))
model.add(layers.SeparableConv2D(32,3,activation='relu', padding='same'))
model.add(layers.MaxPooling2D(2))
model.add(layers.SeparableConv2D(64,3,activation='relu', padding='same'))
model.add(layers.SeparableConv2D(128,3,activation='relu', padding='same'))
model.add(layers.MaxPooling2D(2))
model.add(layers.SeparableConv2D(64,3,activation='relu', padding='same'))
model.add(layers.SeparableConv2D(128,3,activation='relu', padding='same'))
model.add(layers.MaxPooling2D(2))
model.add(layers.Dense(32,activation='relu'))
model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(num_classes,activation='softmax'))

model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_dwsc = model.fit(x_train, y_train, batch_size=128, epochs=10, validation_split=0.1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_separable_conv.py:104: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


Epoch 1/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/step - accuracy: 0.1121 - loss: 2.3016 - val_accuracy: 0.1050 - val_loss: 2.3024
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1132 - loss: 2.3013 - val_accuracy: 0.1050 - val_loss: 2.3021
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1132 - loss: 2.3012 - val_accuracy: 0.1050 - val_loss: 2.3020
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1132 - loss: 2.3012 - val_accuracy: 0.1050 - val_loss: 2.3020
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1132 - loss: 2.3012 - val_accuracy: 0.1050 - val_loss: 2.3019
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1132 - loss: 2.3012 - val_accuracy: 0.1050 - val_loss: 2.3019
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1132 - loss: 2.3012 - val_accuracy: 0.1050 - val_loss: 2.3019
Epoch 8/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1132 - loss: 2.3012 - val_accuracy: 

This is not a best example as MNIST doesn't have any channel to separate.

#### **Hyperparameter Optimization**

When building a deep-learning model, we have to make many seemingly abitrary decisions: How many layers should we stack? how many units or filters ahould go in each layer? Should we use relu as activation, or a different function? Should we use BatchNormalization after a given layer? How much dropout should we use? These can be decided with trial and error. Let's try to automate this.

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

# Define a model-building function for Keras Tuner
def build_tunable_model(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28, 1)))
    model.add(layers.Flatten())

    # Tune the number of hidden layers
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(layers.Dense(units=hp.Int('units_' + str(i),
                                            min_value=32,
                                            max_value=512,
                                            step=32),
                                activation='relu'))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(hp.Float('dropout_' + str(i), min_value=0.0, max_value=0.5, step=0.1)))

    model.add(layers.Dense(10, activation='softmax'))

    # Tune the learning rate for the optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Set up the Keras Tuner RandomSearch
tuner = kt.RandomSearch(
    build_tunable_model,
    objective='val_accuracy',
    max_trials=10,  # Number of hyperparameter combinations to try
    executions_per_trial=2, # Number of models to train for each combination
    directory='mnist_hp_tuning',
    project_name='mnist_classification_hp_opt',
    overwrite=True
)

# Display search space summary
tuner.search_space_summary()

# Run the hyperparameter search
print("\nStarting hyperparameter search...")
tuner.search(x_train, y_train, epochs=5, validation_split=0.1, batch_size=128)

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"\nOptimal number of layers: {best_hps.get('num_layers')}")
for i in range(best_hps.get('num_layers')):
    print(f"Optimal units for layer {i}: {best_hps.get('units_' + str(i))}")
    print(f"Optimal dropout for layer {i}: {best_hps.get('dropout_' + str(i))}")
print(f"Optimal learning rate: {best_hps.get('learning_rate')}")

# Get the best model found during the search
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

# Evaluate the best model on the test data
print("\nEvaluating the best model...")
loss_hp_opt, accuracy_hp_opt = best_model.evaluate(x_test, y_test)
print(f"Test Loss (Hyperparameter Optimized): {loss_hp_opt:.4f}")
print(f"Test Accuracy (Hyperparameter Optimized): {accuracy_hp_opt:.4f}")

Trial 10 Complete [00h 00m 21s]
val_accuracy: 0.9697500169277191

Best val_accuracy So Far: 0.9802500009536743
Total elapsed time: 00h 04m 10s

Optimal number of layers: 2
Optimal units for layer 0: 320
Optimal dropout for layer 0: 0.2
Optimal units for layer 1: 512
Optimal dropout for layer 1: 0.0
Optimal learning rate: 0.001


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 320)            │       251,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 320)            │         1,280 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 320)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 424,010 (1.62 MB)

 Trainable params: 422,346 (1.61 MB)

 Non-trainable params: 1,664 (6.50 KB)


Evaluating the best model...
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9770 - loss: 0.0736
Test Loss (Hyperparameter Optimized): 0.0736
Test Accuracy (Hyperparameter Optimized): 0.9770


Note: One important issue to keep in mind when doing automatic hyperparameter optimization at scale is validation-set overfitting. Because we are updating hyperparameters based on a signal that is computed using the validation data.

#### **Model Ensembling**

Ensembling consist of pooling together the predictions of a set of different models, to produce better predictions.

In [12]:
# Make predictions from individual models on Fashion MNIST test data
pred_a = model.predict(x_test)
pred_b = model_snn.predict(x_test)
pred_c = model_renorm.predict(x_test)

# Combine predictions with weighted averaging (example weights)
# You might need to adjust these weights based on individual model performance
final_preds_fashion_ensemble = (
    0.4 * pred_a +
    0.4 * pred_b +
    0.2 * pred_c 
)

import numpy as np
# Evaluate the ensemble predictions
predicted_labels = np.argmax(final_preds_fashion_ensemble, axis=1)
true_labels = y_test

from sklearn.metrics import accuracy_score
ensemble_accuracy_fashion = accuracy_score(true_labels, predicted_labels)
print(f"\nEnsemble Accuracy on Fashion MNIST: {ensemble_accuracy_fashion:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

Ensemble Accuracy on Fashion MNIST: 0.9842


Note: One thing that is largely not worth doing is ensembling the same network trained several times independently, from different random initializations. If the only difference our models are their random initialization and ther order in which they were exposed to the training data, then our ensemble will be low-diversity and will provide only a tiny improvement over any single model.